In [6]:
import json
import math
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import timm
import torch
import torch.nn as nn
from PIL import Image
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from tqdm import tqdm


# ============================================================
# Configuration
# ============================================================

CSV_PATH = "/content/drive/MyDrive/Colab Notebooks/boneage-training-dataset.csv"
IMAGE_DIR = "/content/drive/MyDrive/Colab Notebooks/cropped_overlayed_RSNA_dataset_1024x1024"
BEST_CHECKPOINT_PATH = "/content/drive/MyDrive/Colab Notebooks/efficientnet_b4_1024x1024_best.pth"
LATEST_CHECKPOINT_PATH = "/content/drive/MyDrive/Colab Notebooks/efficientnet_b4_1024x1024_latest.pth"
PLOT_DIR = "/content/drive/MyDrive/Colab Notebooks/training_plots_efficientnet_b4_1024x1024"

RESUME_TRAINING = True
RESUME_FROM_BEST_IF_NO_LATEST = True
MODEL_NAME = "tf_efficientnet_b4.ns_jft_in1k"
IMAGE_HEIGHT = 1024
IMAGE_WIDTH = 1024
NO_PRETRAINED = False
VAL_SIZE = 0.15
SEED = 42
BATCH_SIZE = 8
GRAD_ACCUM_STEPS = 4
NUM_WORKERS = 8
EPOCHS = 200
STEPS_PER_EPOCH = 250
WARMUP_EPOCHS = 5
PATIENCE = 20
BACKBONE_LR = 1e-4
HEAD_LR = 1e-3
WEIGHT_DECAY = 1e-5
DROP_PATH = 0.1
HEAD_DROPOUT = 0.2
HIDDEN_DIM = 512
SMOOTH_L1_BETA = 6.0
MAX_GRAD_NORM = 1.0
USE_AMP = True
SUPPORTED_IMAGE_EXTENSIONS = [".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"]

# ============================================================
# Reproducibility
# ============================================================

def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

# ============================================================
# Image indexing / CSV filtering
# ============================================================

def normalize_id(value) -> str:
    if pd.isna(value): return ""
    if isinstance(value, float) and value.is_integer(): return str(int(value))
    return str(value).strip()

def build_image_index(image_dir):
    image_dir = Path(image_dir)
    if not image_dir.exists(): raise FileNotFoundError(f"IMAGE_DIR does not exist: {image_dir}")
    image_index = {}
    for ext in SUPPORTED_IMAGE_EXTENSIONS:
        for path in image_dir.glob(f"*{ext}"): image_index[path.stem] = path
    return image_index

def filter_dataframe_to_existing_images(df, image_index, plot_dir):
    df = df.copy()
    df["id"] = df["id"].apply(normalize_id)
    exists_mask = df["id"].isin(image_index.keys())
    missing_df = df.loc[~exists_mask].copy()
    filtered_df = df.loc[exists_mask].copy()
    plot_dir = Path(plot_dir)
    plot_dir.mkdir(parents=True, exist_ok=True)
    if len(missing_df) > 0:
        missing_path = plot_dir / "missing_images.csv"
        missing_df.to_csv(missing_path, index=False)
        print(f"Warning: {len(missing_df)} rows missing images.")
    return filtered_df

# ============================================================
# Dataset / Model
# ============================================================

class BoneAgeDataset(Dataset):
    def __init__(self, dataframe, image_index, image_height, image_width):
        self.df = dataframe.reset_index(drop=True).copy()
        self.image_index = image_index
        self.transform = transforms.Compose([
            transforms.Resize((image_height, image_width)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ])
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image_path = self.image_index[normalize_id(row["id"])]
        image = self.transform(Image.open(image_path).convert("RGB"))
        boneage = torch.tensor(float(row["boneage"]), dtype=torch.float32)
        male = torch.tensor([float(str(row["male"]).lower() == "true")], dtype=torch.float32)
        return {"image": image, "male": male, "target": boneage, "id": row["id"]}

class BoneAgeEfficientNet(nn.Module):
    def __init__(self, model_name, pretrained=True, drop_path_rate=0.1, head_dropout=0.2, hidden_dim=512):
        super().__init__()
        self.backbone = timm.create_model(model_name, pretrained=pretrained, num_classes=0, global_pool="avg", drop_path_rate=drop_path_rate)
        feature_dim = self.backbone.num_features
        self.regression_head = nn.Sequential(
            nn.Linear(feature_dim + 1, hidden_dim), nn.SiLU(), nn.Dropout(head_dropout), nn.Linear(hidden_dim, 1)
        )
    def forward(self, image, male):
        features = self.backbone(image)
        return self.regression_head(torch.cat([features, male], dim=1)).squeeze(1)

# ============================================================
# Split / Training / Metrics
# ============================================================

def create_stratified_split(df, val_size, seed):
    df = df.copy()
    df["age_bin"] = pd.qcut(df["boneage"], q=10, duplicates="drop", labels=False)
    df["stratify_col"] = df["age_bin"].astype(str) + "_" + df["male"].astype(str)
    train_df, val_df = train_test_split(df, test_size=val_size, random_state=seed, stratify=df["stratify_col"])
    return train_df.drop(columns=["age_bin", "stratify_col"]), val_df.drop(columns=["age_bin", "stratify_col"])

def compute_mae(preds, targets): return float(np.mean(np.abs(np.asarray(preds) - np.asarray(targets))))
def compute_rmse(preds, targets): return float(np.sqrt(np.mean((np.asarray(preds) - np.asarray(targets))**2)))

def train_one_epoch(model, loader, criterion, optimizer, device, scaler, use_amp, grad_accum_steps, max_grad_norm, steps_per_epoch=None, loader_iter=None):
    model.train()
    all_preds, all_targets = [], []
    optimizer.zero_grad(set_to_none=True)
    num_steps = steps_per_epoch if steps_per_epoch else len(loader)
    progress_bar = tqdm(range(num_steps), desc="Training", leave=False)
    for step_idx in progress_bar:
        try: batch = next(loader_iter)
        except: loader_iter = iter(loader); batch = next(loader_iter)
        images, males, targets = batch["image"].to(device), batch["male"].to(device), batch["target"].to(device)
        with torch.amp.autocast(device_type="cuda", enabled=use_amp):
            preds = model(images, males)
            loss = criterion(preds, targets) / grad_accum_steps
        if scaler: scaler.scale(loss).backward()
        else: loss.backward()
        if (step_idx + 1) % grad_accum_steps == 0:
            if scaler: scaler.unscale_(optimizer); torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm); scaler.step(optimizer); scaler.update()
            else: torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm); optimizer.step()
            optimizer.zero_grad(set_to_none=True)
        all_preds.extend(preds.detach().cpu().numpy()); all_targets.extend(targets.detach().cpu().numpy())
        progress_bar.set_postfix({"mae": f"{compute_mae(all_preds, all_targets):.2f}"})
    return compute_mae(all_preds, all_targets), loader_iter

@torch.no_grad()
def validate_one_epoch(model, loader, device, use_amp):
    model.eval()
    all_preds, all_targets, all_males, all_ids = [], [], [], []
    for batch in tqdm(loader, desc="Validation", leave=False):
        images, males, targets = batch["image"].to(device), batch["male"].to(device), batch["target"].to(device)
        with torch.amp.autocast(device_type="cuda", enabled=use_amp): preds = model(images, males)
        all_preds.extend(preds.cpu().numpy()); all_targets.extend(targets.cpu().numpy()); all_males.extend(males.cpu().numpy().flatten()); all_ids.extend(batch["id"])
    res_df = pd.DataFrame({"id": all_ids, "target": all_targets, "prediction": all_preds, "male": all_males, "abs_error": np.abs(np.array(all_preds)-np.array(all_targets))})
    return compute_mae(all_preds, all_targets), compute_rmse(all_preds, all_targets), res_df

# ============================================================
# Plotting (Simplified for Report)
# ============================================================

def save_plots(history_df, predictions_df, plot_dir, prefix="latest"):
    plot_dir = Path(plot_dir); plot_dir.mkdir(parents=True, exist_ok=True)
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    # MAE Curves
    axes[0].plot(history_df["epoch"], history_df["train_mae"], label="Train MAE", marker='o')
    axes[0].plot(history_df["epoch"], history_df["val_mae"], label="Val MAE", marker='o')
    axes[0].set_title("Mean Absolute Error (Months)"); axes[0].set_xlabel("Epoch"); axes[0].legend()
    # Regression Plot
    axes[1].scatter(predictions_df["target"], predictions_df["prediction"], alpha=0.4, s=10)
    lims = [min(predictions_df["target"].min(), predictions_df["prediction"].min()), max(predictions_df["target"].max(), predictions_df["prediction"].max())]
    axes[1].plot(lims, lims, 'r--', alpha=0.75, zorder=0); axes[1].set_title("Predicted vs. True Bone Age"); axes[1].set_xlabel("True Age"); axes[1].set_ylabel("Predicted Age")
    plt.tight_layout(); plt.savefig(plot_dir / f"{prefix}_report_plot.png", dpi=200); plt.close()

# Main execution logic remains similar but stripped of verbose loss prints
def main():
    set_seed(SEED)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    image_index = build_image_index(IMAGE_DIR)
    df = filter_dataframe_to_existing_images(pd.read_csv(CSV_PATH), image_index, PLOT_DIR)
    train_df, val_df = create_stratified_split(df, VAL_SIZE, SEED)
    train_loader = DataLoader(BoneAgeDataset(train_df, image_index, IMAGE_HEIGHT, IMAGE_WIDTH), batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
    val_loader = DataLoader(BoneAgeDataset(val_df, image_index, IMAGE_HEIGHT, IMAGE_WIDTH), batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
    model = BoneAgeEfficientNet(MODEL_NAME, pretrained=not NO_PRETRAINED, drop_path_rate=DROP_PATH, head_dropout=HEAD_DROPOUT, hidden_dim=HIDDEN_DIM).to(device)
    optimizer = torch.optim.AdamW([{'params': [p for n, p in model.named_parameters() if "head" not in n], 'lr': BACKBONE_LR}, {'params': model.regression_head.parameters(), 'lr': HEAD_LR}], weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
    criterion = nn.SmoothL1Loss(beta=SMOOTH_L1_BETA)
    scaler = torch.amp.GradScaler("cuda") if USE_AMP else None

    history = []; best_mae = float('inf'); loader_iter = None
    for epoch in range(1, EPOCHS + 1):
        t_mae, loader_iter = train_one_epoch(model, train_loader, criterion, optimizer, device, scaler, USE_AMP, GRAD_ACCUM_STEPS, MAX_GRAD_NORM, STEPS_PER_EPOCH, loader_iter)
        v_mae, v_rmse, v_df = validate_one_epoch(model, val_loader, device, USE_AMP)
        scheduler.step()
        history.append({"epoch": epoch, "train_mae": t_mae, "val_mae": v_mae})
        print(f"Epoch {epoch} | Train MAE: {t_mae:.2f} | Val MAE: {v_mae:.2f}")
        if v_mae < best_mae:
            best_mae = v_mae; torch.save(model.state_dict(), BEST_CHECKPOINT_PATH)
            save_plots(pd.DataFrame(history), v_df, PLOT_DIR, "best")

if __name__ == "__main__":
    main()

Using device: cuda
Found image files: 12608
Total CSV samples: 12611
Bone age range: 1 to 228 months
Male distribution:
male
True     6833
False    5778
Name: count, dtype: int64
Missing image IDs saved to: /content/drive/MyDrive/Colab Notebooks/training_plots_efficientnet_b4_1024x1024/missing_images.csv
First missing IDs: ['1435', '1446', '1814', '2178', '2419', '2934', '3079', '3100', '3991', '4270', '5530', '6232', '6319', '7308', '7893', '8580', '8757', '8940', '9969', '10441']
Samples after image-file filtering: 12584
Train samples: 10696
Validation samples: 1888
Model: tf_efficientnet_b4.ns_jft_in1k
Image size: 1024x1024 width x height
Effective batch size: 32
Total parameters: 18,467,657
Trainable parameters: 18,467,657
No checkpoint loaded. Starting from scratch.

Epoch 1/200


Train loss: 113.5978 | Train MAE: 116.60 months | Val loss: 94.4016 | Val MAE: 97.39 months | Val RMSE: 105.72 months | Val Median AE: 102.94 months | Val Mean Error: -96.63 months | LR: 4.00e-05
Saved new best model to: /content/drive/MyDrive/Colab Notebooks/efficientnet_b4_1024x1024_best.pth
Best Val MAE: 97.39 months

Epoch 2/200


Train loss: 57.8622 | Train MAE: 60.80 months | Val loss: 31.0395 | Val MAE: 33.99 months | Val RMSE: 38.77 months | Val Median AE: 33.00 months | Val Mean Error: -33.42 months | LR: 6.00e-05
Saved new best model to: /content/drive/MyDrive/Colab Notebooks/efficientnet_b4_1024x1024_best.pth
Best Val MAE: 33.99 months

Epoch 3/200


Train loss: 16.2979 | Train MAE: 19.11 months | Val loss: nan | Val MAE: nan months | Val RMSE: nan months | Val Median AE: nan months | Val Mean Error: nan months | LR: 8.00e-05
No improvement. Patience: 1/20

Epoch 4/200


Train loss: 14.6760 | Train MAE: 17.46 months | Val loss: 11.2304 | Val MAE: 13.96 months | Val RMSE: 17.87 months | Val Median AE: 12.00 months | Val Mean Error: -9.75 months | LR: 1.00e-04
Saved new best model to: /content/drive/MyDrive/Colab Notebooks/efficientnet_b4_1024x1024_best.pth
Best Val MAE: 13.96 months

Epoch 5/200


...

Train loss: 3.7166 | Train MAE: 6.09 months | Val loss: 5.0180 | Val MAE: 7.50 months | Val RMSE: 9.71 months | Val Median AE: 6.06 months | Val Mean Error: -2.99 months | LR: 8.22e-05
No improvement. Patience: 16/20

Epoch 60/200


Train loss: 3.1364 | Train MAE: 5.43 months | Val loss: 4.8886 | Val MAE: 7.35 months | Val RMSE: 9.59 months | Val Median AE: 6.00 months | Val Mean Error: 0.16 months | LR: 8.16e-05
No improvement. Patience: 17/20

Epoch 61/200


Train loss: 3.0022 | Train MAE: 5.29 months | Val loss: 5.9869 | Val MAE: 8.54 months | Val RMSE: 10.89 months | Val Median AE: 7.20 months | Val Mean Error: -5.69 months | LR: 8.10e-05
No improvement. Patience: 18/20

Epoch 62/200


Train loss: 3.2706 | Train MAE: 5.57 months | Val loss: 5.3614 | Val MAE: 7.86 months | Val RMSE: 10.21 months | Val Median AE: 6.46 months | Val Mean Error: 3.58 months | LR: 8.04e-05
No improvement. Patience: 19/20

Epoch 63/200


Train loss: 3.2433 | Train MAE: 5.56 months | Val loss: 4.9902 | Val MAE: 7.47 months | Val RMSE: 9.72 months | Val Median AE: 6.00 months | Val Mean Error: -1.35 months | LR: 7.97e-05
No improvement. Patience: 20/20
Early stopping triggered. Best epoch: 43, Best Val MAE: 7.06 months

Training finished.


In [3]:
import os

print(os.listdir('/content/drive/MyDrive/Colab Notebooks'))
print(os.cpu_count())

['cropped_overlayed_RSNA_dataset', 'cropped_overlayed_RSNA_dataset_1024x1024', 'train_efficientnetB4_512.ipynb', 'boneage-training-dataset.csv', 'convnextv2_huge_22k_512_ema.pt', 'overlayed_RSNA_dataset', 'training_plots', 'resnet50_best.pth', 'training_plots_resnet50_448', 'resnet50_448_best.pth', 'training_plots_convnextv2_base_512', 'training_plots_convnextv2_base_384x512', 'train_convnextv2_512x384.ipynb', 'training_plots_efficientnet_b4_512x512', 'convnextv2_base_384x512_best.pth', 'resnet50_train.ipynb', 'convnextv2_base_384x512_latest.pth', 'efficientnet_b4_512x512_best.pth', 'efficientnet_b4_512x512_latest.pth', 'resnet50_train_448.ipynb', 'training_plots_efficientnet_b4_1024x1024']
12


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
# drive.mount('/content/drive', force_remount=True)

In [ ]:
!nvidia-smi